In [17]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
from functools import partial
from jax import flatten_util
from VMC_tool import hi, edges,ha,SingleStateAnsatz,create_machine,compute_local_energies,\
    compute_qgt,forces_expect_hermitian

In [25]:
import jax
import jax.numpy as jnp
from functools import partial

def make_get_all_next_states(edges):
    edges = tuple(tuple(e) for e in edges)

    @jax.jit
    def get_all_next_states_jit(S: jnp.ndarray):
        """
        适配多链：S形状从 (n_orbitals,) → (n_chains, n_orbitals)
        """
        next_states = []
        masks = []

        for (i, j) in edges:
            occ_i = S[..., i]  # 多链维度：(n_chains,)
            occ_j = S[..., j]
            valid = (occ_i == 1) & (occ_j == 0) | (occ_i == 0) & (occ_j == 1)
            # 对每条链单独翻转i/j位
            new_state = S.at[..., i].set(occ_j).at[..., j].set(occ_i)
            next_states.append(new_state)
            masks.append(valid)

        # 输出形状：(n_edges, n_chains, n_orbitals) 和 (n_edges, n_chains)
        return jnp.stack(next_states), jnp.stack(masks)
    
    return get_all_next_states_jit

# ==============================
# 改造1：MH步骤适配多链
# ==============================
def make_metropolis_hastings_step(edges, machine, params):
    get_all_next = make_get_all_next_states(edges)
    
    @jax.jit
    def metropolis_hastings_step_jit(S: jnp.ndarray, key: jax.Array):
        """
        S: (n_chains, n_orbitals) → 多链状态
        key: 随机数种子（每条链独立拆分）
        """
        n_chains = S.shape[0]
        # 1. 生成候选状态：(n_edges, n_chains, n_orbitals)
        candidates, valid_mask = get_all_next(S)  # valid_mask: (n_edges, n_chains)
        
        # 2. 每条链独立选候选（避免所有链选同一个edge）
        key, subk = jax.random.split(key)
        subkeys = jax.random.split(subk, n_chains)  # (n_chains, 2)
        # 对每条链采样候选索引
        idx = jax.vmap(lambda k: jax.random.choice(k, candidates.shape[0]))(subkeys)  # (n_chains,)
        
        # 3. 按索引取候选状态（多链）
        # 先构造索引：(n_chains,) → (n_chains, 2) (edge_idx, chain_idx)
        chain_idx = jnp.arange(n_chains)
        S_cand = candidates[idx, chain_idx]  # (n_chains, n_orbitals)
        is_valid = valid_mask[idx, chain_idx]  # (n_chains,)

        # 4. 计算接受率（多链并行）
        log_psi_curr = machine(params, S)  # (n_chains,)
        log_psi_cand = machine(params, S_cand)  # (n_chains,)
        log_accept_ratio = 2 * jnp.real(log_psi_cand - log_psi_curr)  # (n_chains,)

        # 5. 每条链独立判断是否接受
        key, subk = jax.random.split(key)
        subkeys = jax.random.split(subk, n_chains)
        u = jax.vmap(lambda k: jax.random.uniform(k))(subkeys)  # (n_chains,)
        accept = is_valid & (log_accept_ratio > jnp.log(u))  # (n_chains,)

        # 6. 更新每条链的状态
        S_new = jnp.where(accept[:, None], S_cand, S)  # 广播accept到(n_chains, n_orbitals)
        return S_new, accept, key

    return metropolis_hastings_step_jit

# ==============================
# 改造2：多链采样器核心
# ==============================
@partial(jax.jit, static_argnums=(0, 1, 2, 4,5))
def mcmc_sampler_multichain(
    n_samples_per_chain: int,  # 每条链采样数
    n_warmup: int,
    initial_states: jnp.ndarray,  # (n_chains, n_orbitals) → 多链初始状态
    edges: tuple[tuple[int, int]],
    machine: callable,
    params: jnp.ndarray,
    seed: int = 42
):
    key = jax.random.PRNGKey(seed)
    # 绑定params到MH步骤
    mh_step = make_metropolis_hastings_step(edges, machine, params)

    # 预烧：多链并行warmup
    def warmup_loop(carry, _):
        states, rng = carry
        states, _, rng = mh_step(states, rng)
        return (states, rng), None

    (current_states, key), _ = jax.lax.scan(
        warmup_loop,
        (initial_states, key),
        xs=None,
        length=n_warmup
    )

    # 采样：多链并行采样
    def sample_loop(carry, _):
        states, rng = carry
        states, accepted, rng = mh_step(states, rng)
        return (states, rng), states

    (_, _), samples = jax.lax.scan(
        sample_loop,
        (current_states, key),
        xs=None,
        length=n_samples_per_chain
    )
    # samples形状：(n_samples_per_chain, n_chains, n_orbitals)
    # 展平为 (n_samples_total, n_orbitals)，其中n_samples_total = n_samples_per_chain * n_chains
    samples = samples.reshape(-1, initial_states.shape[-1])
    return samples

# ==============================
# 改造3：生成多链随机初始状态（模拟NetKet的默认行为）
# ==============================
def generate_random_initial_states(hilbert, n_chains: int, seed: int = 42):
    """
    模仿NetKet：从希尔伯特空间随机生成多链初始状态
    hilbert: NetKet的SpinOrbitalFermions希尔伯特空间
    n_chains: 链数
    """
    key = jax.random.PRNGKey(seed)
    # 希尔伯特空间的随机采样（NetKet内部逻辑）
    return hilbert.random_state(key, n_chains)

In [30]:
machine, graphdef, params = create_machine(model)
samples = mcmc_sampler(n_samples_per_chain=200,
                            n_warmup=100,
                            initial_states=generate_random_initial_states(hi,16,12),
                            edges=((0, 1), (2, 3)),
                            machine=machine,
                            params=params,
                            seed=21
                            )

samples.shape

(3200, 4)

In [31]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.01)  # 学习率 0.01
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 300  # 迭代次数
N_SAMPLES = 1008  # 样本数

# ===================== 7. 训练循环 =====================
print("\n" + "="*60)
print("开始纯 JAX VMC 训练 (自然梯度下降法)")
print("="*60)

# 用于记录训练历史
history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}

for step in range(N_ITER):
    # 1. 采样
    samples = mcmc_sampler(n_samples_per_chain=N_SAMPLES,
                             n_warmup=100,
                             initial_states=generate_random_initial_states(hi,4,12),
                             edges=((0, 1), (2, 3)),
                             machine=machine,
                             params=params,
                             seed=21
                             )
    #samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x*2, grad)
    #qgt_reg, unravel_fn = compute_qgt(machine,params,samples.reshape(-1,4),0.001)
    qgt_reg,qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001) 
    grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    #自然梯度 natural-gradient = S^{-1} * grad
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
        
    # 4. 更新参数（自然梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)



开始纯 JAX VMC 训练 (自然梯度下降法)
Step   0 | E: -0.49602383 ± 0.003761 | FCI: -1.01546825 | Error: 0.519444
Step  50 | E: -0.96351647 ± 0.002600 | FCI: -1.01546825 | Error: 0.051952
Step 100 | E: -1.00910713 ± 0.000581 | FCI: -1.01546825 | Error: 0.006361
Step 150 | E: -1.01518036 ± 0.000112 | FCI: -1.01546825 | Error: 0.000288
Step 200 | E: -1.01255540 ± 0.000253 | FCI: -1.01546825 | Error: 0.002913
Step 250 | E: -1.01328571 ± 0.000720 | FCI: -1.01546825 | Error: 0.002183
Step 299 | E: -1.01525737 ± 0.000093 | FCI: -1.01546825 | Error: 0.000211

训练完成!
最终能量：-1.01559204 ± 0.000127 Ha
FCI 基准：-1.01546825 Ha
绝对误差：0.000124 Ha
相对误差：0.0122%
